# Notebook 01 — Telemetry Generation & Latent Behavioral Structures
## Latent Behavioral State Machines (LBSM): Manifold Geometry of Adaptive Agent Telemetry

---

**Purpose** — This notebook is the *experimental birth* of the paper.  
It answers the foundational question:

> *Does meaningful latent behavioral structure actually appear in simulated agent telemetry?*

**Responsibilities**
1. **Generate** a large multi-agent telemetry dataset via the LBSM simulation core  
2. **Inspect** distributional statistics: means, variances, state frequencies  
3. **Visualize** behavioral signatures: latency vs entropy, state distributions, separability  
4. **Observe** latent structure: Do clusters emerge? Do regimes form geometrically separable regions?

**Reference** — *"Latent Behavioral State Machines: Manifold Geometry of Adaptive Agent Telemetry"*, §4 — Experimental Setup & Dataset Construction

---


## 1. Imports & Environment

In [1]:
from __future__ import annotations

# ── Standard library
import warnings
import logging

# ── Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ── Simulation core
from src.simulation import (
    TelemetryGenerator,
    BEHAVIOR_PROFILES,
    PROFILE_NAMES,
    TELEMETRY_FEATURES,
    profile_distance_matrix,
    regime_separability_ratio,
)
from src.simulation.behavior_profiles import BehaviorProfile

# ── Aesthetic configuration
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

PALETTE: dict[str, str] = {name: BEHAVIOR_PROFILES[name].color for name in PROFILE_NAMES}
REGIME_ORDER = list(PROFILE_NAMES)           # stable, exploratory, adaptive, unstable

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi":        130,
    "figure.facecolor":  "white",
    "axes.facecolor":    "#fafafa",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "DejaVu Sans",
})

SEED = 42
RNG  = np.random.default_rng(SEED)

print("Environment ready.")
print(f"Regimes          : {PROFILE_NAMES}")
print(f"Telemetry features: {TELEMETRY_FEATURES}")


Environment ready.
Regimes          : ('stable', 'exploratory', 'adaptive', 'unstable')
Telemetry features: ('latency', 'entropy', 'reward', 'memory_usage', 'error_rate', 'action_freq')


## 2. Dataset Generation

We instantiate a `TelemetryGenerator` with **20 heterogeneous agents** running for **2 000 time-steps** each,
yielding **40 000 observations**. Seeds ensure full reproducibility.

Each agent's hidden state evolves as a first-order Markov chain; observable telemetry is emitted
from the corresponding regime profile via an AR(1) process (see `behavior_profiles.py §BehaviorProfile.sample`).


In [2]:
N_AGENTS     = 20
N_TIMESTEPS  = 2_000

gen = TelemetryGenerator(
    n_agents    = N_AGENTS,
    n_timesteps = N_TIMESTEPS,
    seed        = SEED,
    verbose     = False,           # suppress logs inside notebook
)

df = gen.run()

print(f"Dataset shape    : {df.shape}")
print(f"Columns          : {df.columns.tolist()}")
print(f"Memory usage     : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print()
print(df.head(6).to_string(index=False))


Dataset shape    : (40000, 17)
Columns          : ['agent_id', 'timestep', 'hidden_state', 'latency', 'entropy', 'reward', 'memory_usage', 'error_rate', 'action_freq', 'state_label', 'is_anomaly', 'latency_z', 'entropy_z', 'reward_z', 'memory_usage_z', 'error_rate_z', 'action_freq_z']
Memory usage     : 8667.5 KB

  agent_id  timestep hidden_state    latency  entropy   reward  memory_usage  error_rate  action_freq  state_label  is_anomaly  latency_z  entropy_z  reward_z  memory_usage_z  error_rate_z  action_freq_z
agent_0000         0       stable  52.437737 0.675202 8.950271    134.108471    0.004392    12.395641            0           0  -0.633774  -0.951322  0.988841       -0.598758     -0.756283      -0.809722
agent_0000         1  exploratory 110.512722 2.790759 3.976347    230.778929    0.131112    25.396184            1           0  -0.167114   0.431582 -0.690320        0.317016      0.142336       0.018114
agent_0000         2     adaptive  94.350187 1.299248 7.168751    136.02

## 3. Descriptive Statistics

### 3.1 Per-regime means & standard deviations
Ground-truth regime statistics (from `behavior_profiles.py`) versus empirical measurements from the simulation.
A close match validates the emission model.


In [3]:
stats = gen.summary_statistics()
pd.options.display.float_format = "{:.3f}".format

print("=== Empirical per-regime statistics ===")
print(stats.to_string())


=== Empirical per-regime statistics ===
             latency                       entropy                   reward                     memory_usage                        error_rate                   action_freq                     
                mean     std   min     max    mean   std   min   max   mean   std    min    max         mean    std     min     max       mean   std   min   max        mean    std   min     max
hidden_state                                                                                                                                                                                     
adaptive      84.399  21.376 4.890 160.709   1.595 0.377 0.260 3.354  6.813 1.095  2.430 10.506      159.641 26.992  51.821 260.108      0.061 0.027 0.000 0.170      19.908  4.344 3.376  36.670
exploratory  119.983  31.472 9.893 235.353   2.810 0.577 0.564 4.925  5.013 1.266  0.158  9.734      200.009 36.701  54.212 325.446      0.100 0.041 0.000 0.250      24.966  6.195 0.00

### 3.2 State frequencies

In [4]:
freq = gen.state_frequencies()
print("Global state frequencies (fraction of total timesteps):")
print(freq.to_string())
print()

# pi is a property of the shared transition matrix T, not of any individual
# agent — every agent uses the same DEFAULT_TRANSITION_MATRIX (see B6).
pi_theory = gen._agents[0].stationary_distribution()
print("Theoretical stationary distribution (of the shared transition matrix T):")
for name, p in zip(PROFILE_NAMES, pi_theory):
    print(f"  {name:<12s}: {p:.4f}")


Global state frequencies (fraction of total timesteps):
hidden_state
stable        0.378
exploratory   0.213
adaptive      0.210
unstable      0.199

Theoretical stationary distribution (of the shared transition matrix T):
  stable      : 0.3734
  exploratory : 0.2158
  adaptive    : 0.2075
  unstable    : 0.2033


### 3.3 Separability ratio (univariate Fisher criterion)

In [5]:
# Empirical separability ratio, computed directly from the simulated
# telemetry (not from BEHAVIOR_PROFILES config — see error register B1).
means_emp = df.groupby("hidden_state")[list(TELEMETRY_FEATURES)].mean()
stds_emp  = df.groupby("hidden_state")[list(TELEMETRY_FEATURES)].std()

grand_mean  = means_emp.mean(axis=0)
between_var = ((means_emp - grand_mean) ** 2).mean(axis=0)
within_var  = (stds_emp ** 2).mean(axis=0)
sep = (between_var / (within_var + 1e-12)).to_dict()

sep_df = pd.DataFrame.from_dict(sep, orient="index", columns=["separability_ratio"])
sep_df = sep_df.sort_values("separability_ratio", ascending=False)
print("Empirical feature separability ratios (higher = more discriminative):")
print(sep_df.to_string())
print()
print(f"Most discriminative feature: {sep_df.index[0]}  ({sep_df.iloc[0,0]:.2f})")
print(f"Least discriminative feature: {sep_df.index[-1]}  ({sep_df.iloc[-1,0]:.2f})")


Empirical feature separability ratios (higher = more discriminative):
              separability_ratio
memory_usage               4.512
entropy                    4.024
latency                    3.352
reward                     2.697
error_rate                 2.648
action_freq                1.898

Most discriminative feature: memory_usage  (4.51)
Least discriminative feature: action_freq  (1.90)


### 3.4 Inter-regime centroid distances

In [6]:
# Empirical pairwise centroid distances, computed directly from the
# simulated telemetry (not from BEHAVIOR_PROFILES config — see B2).
means_emp_ord = means_emp.reindex(PROFILE_NAMES)
D = np.zeros((len(PROFILE_NAMES), len(PROFILE_NAMES)))
for i, a in enumerate(PROFILE_NAMES):
    for j, b in enumerate(PROFILE_NAMES):
        D[i, j] = np.linalg.norm(means_emp_ord.loc[a] - means_emp_ord.loc[b])

dist_df = pd.DataFrame(D, index=PROFILE_NAMES, columns=PROFILE_NAMES)
print("Empirical pairwise Euclidean distances between regime centroids (raw feature space):")
print(dist_df.round(2).to_string())
print()
print("Largest separation:", dist_df.stack().idxmax(), "→", dist_df.stack().max().round(2))


Empirical pairwise Euclidean distances between regime centroids (raw feature space):
             stable  exploratory  adaptive  unstable
stable        0.000      106.760    52.670   397.270
exploratory 106.760        0.000    54.090   291.790
adaptive     52.670       54.090     0.000   345.180
unstable    397.270      291.790   345.180     0.000

Largest separation: ('stable', 'unstable') → 397.27


### 3.5 Per-agent summary

In [7]:
agent_stats = gen.per_agent_statistics()
print(agent_stats.to_string())


           dominant_state  mean_reward  mean_error  mean_latency  n_transitions
agent_id                                                                       
agent_0000         stable        6.089       0.106       128.512            693
agent_0001         stable        6.150       0.105       128.987            737
agent_0002         stable        6.118       0.108       123.560            725
agent_0003         stable        5.959       0.113       131.846            718
agent_0004         stable        6.196       0.104       122.623            703
agent_0005         stable        6.167       0.104       122.392            683
agent_0006         stable        5.729       0.121       144.773            668
agent_0007         stable        5.746       0.122       141.215            718
agent_0008         stable        6.116       0.112       130.726            697
agent_0009         stable        6.045       0.110       134.189            716
agent_0010         stable        6.130  

## 4. Visualization

### 4.1 State frequency bar chart
How much time did the simulated pool collectively spend in each behavioral regime?


In [8]:
fig, ax = plt.subplots(figsize=(7, 3.5))

bars = ax.bar(
    REGIME_ORDER,
    [freq[r] for r in REGIME_ORDER],
    color=[PALETTE[r] for r in REGIME_ORDER],
    edgecolor="white",
    linewidth=1.4,
    width=0.55,
)

for bar, regime in zip(bars, REGIME_ORDER):
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + 0.005,
        f"{h:.3f}",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

ax.set_ylabel("Fraction of total observations", fontsize=11)
ax.set_xlabel("Behavioral regime", fontsize=11)
ax.set_title("Global State Frequencies across Agent Pool", fontsize=13, fontweight="bold", pad=12)
ax.set_ylim(0, max(freq.values) + 0.08)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.tight_layout()
plt.savefig("fig01_state_frequencies.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig01_state_frequencies.png")


Saved: fig01_state_frequencies.png


### 4.2 Per-feature distribution by regime (violin + strip)

The most important distributional view: can we *visually* separate the four regimes along each feature axis?
Overlapping violins → weak discriminative power. Non-overlapping → strong.


In [9]:
n_feats = len(TELEMETRY_FEATURES)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, feat in zip(axes, TELEMETRY_FEATURES):
    # Violin
    parts = ax.violinplot(
        [df.loc[df["hidden_state"] == r, feat].values for r in REGIME_ORDER],
        positions=range(len(REGIME_ORDER)),
        showmedians=True,
        showextrema=False,
        widths=0.7,
    )
    for i, (pc, regime) in enumerate(zip(parts["bodies"], REGIME_ORDER)):
        pc.set_facecolor(PALETTE[regime])
        pc.set_alpha(0.55)
    parts["cmedians"].set_color("black")
    parts["cmedians"].set_linewidth(1.8)

    # Ground-truth mean marker
    for i, regime in enumerate(REGIME_ORDER):
        gt_mean = BEHAVIOR_PROFILES[regime].means[list(TELEMETRY_FEATURES).index(feat)]
        ax.scatter(i, gt_mean, marker="D", s=42, color="black", zorder=5, label="GT mean" if i == 0 else "")

    ax.set_xticks(range(len(REGIME_ORDER)))
    ax.set_xticklabels(REGIME_ORDER, rotation=20, ha="right", fontsize=9)
    ax.set_title(feat, fontsize=11, fontweight="bold")
    ax.set_ylabel("value")

axes[0].legend(loc="upper left", fontsize=8)

fig.suptitle(
    "Feature Distributions by Behavioral Regime\n(violin = empirical, ◆ = ground-truth mean)",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig("fig02_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig02_feature_distributions.png")


Saved: fig02_feature_distributions.png


### 4.3 Latency vs Entropy scatter — the primary behavioral plane

These two features are theoretically orthogonal in meaning:
- **Latency** → computational / reactive load  
- **Entropy** → policy stochasticity  

Do they jointly separate the four regimes?


In [10]:
fig, ax = plt.subplots(figsize=(8.5, 6.5))

sample_frac = 0.20   # plot 20% to avoid overplotting
df_sample = pd.concat([
    df[df["hidden_state"]==r].sample(frac=sample_frac, random_state=SEED)
    for r in REGIME_ORDER
])

for regime in REGIME_ORDER:
    sub = df_sample[df_sample["hidden_state"] == regime]
    ax.scatter(
        sub["latency"], sub["entropy"],
        c=PALETTE[regime], alpha=0.30, s=14, linewidths=0,
        label=regime,
    )

# Ground-truth centroids with error ellipses (1σ)
for regime in REGIME_ORDER:
    bp = BEHAVIOR_PROFILES[regime]
    lat_idx = list(TELEMETRY_FEATURES).index("latency")
    ent_idx = list(TELEMETRY_FEATURES).index("entropy")
    cx, cy = bp.means[lat_idx], bp.means[ent_idx]
    sx, sy = bp.stds[lat_idx],  bp.stds[ent_idx]

    ellipse = plt.matplotlib.patches.Ellipse(
        (cx, cy), width=2*sx, height=2*sy,
        edgecolor=PALETTE[regime], facecolor="none",
        linewidth=2.5, linestyle="--", zorder=4,
    )
    ax.add_patch(ellipse)
    ax.scatter(cx, cy, marker="*", s=220, color=PALETTE[regime],
               edgecolors="black", linewidths=0.7, zorder=5)

ax.set_xlabel("Latency (ms)", fontsize=12)
ax.set_ylabel("Entropy (bits)", fontsize=12)
ax.set_title(
    "Latency vs Entropy: Primary Behavioral Plane\n"
    "(scatter = 20% sample · ★ = regime centroid · dashed = 1σ ellipse)",
    fontsize=12, fontweight="bold", pad=10,
)

legend_handles = [
    mpatches.Patch(color=PALETTE[r], label=r.capitalize()) for r in REGIME_ORDER
]
ax.legend(handles=legend_handles, title="Regime", fontsize=10, title_fontsize=10)

plt.tight_layout()
plt.savefig("fig03_latency_entropy_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig03_latency_entropy_scatter.png")


Saved: fig03_latency_entropy_scatter.png


### 4.4 Feature separability heatmap

Which features most strongly discriminate across regimes?  
(Between-class / within-class variance — univariate Fisher criterion)


In [11]:
sep_values = np.array([sep[f] for f in TELEMETRY_FEATURES])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Left: bar chart
ax = axes[0]
colors_bar = plt.cm.RdYlGn(np.linspace(0.25, 0.85, len(TELEMETRY_FEATURES)))
sorted_idx = np.argsort(sep_values)[::-1]
sorted_feats  = [TELEMETRY_FEATURES[i] for i in sorted_idx]
sorted_values = sep_values[sorted_idx]

bars = ax.barh(sorted_feats[::-1], sorted_values[::-1],
               color=colors_bar, edgecolor="white", height=0.6)
for bar, val in zip(bars, sorted_values[::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Separability ratio (Fisher criterion)", fontsize=10)
ax.set_title("Univariate Feature Separability", fontsize=11, fontweight="bold")
ax.set_xlim(0, sorted_values.max() * 1.18)

# ── Right: per-regime mean heatmap (z-scored)
ax = axes[1]
means_matrix = np.stack([BEHAVIOR_PROFILES[n].means for n in REGIME_ORDER])
# Normalize each feature to [0,1] for visual clarity
means_norm = (means_matrix - means_matrix.min(0)) / ((means_matrix.max(0) - means_matrix.min(0)) + 1e-9)

im = ax.imshow(means_norm, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(TELEMETRY_FEATURES)))
ax.set_xticklabels(TELEMETRY_FEATURES, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(REGIME_ORDER)))
ax.set_yticklabels([r.capitalize() for r in REGIME_ORDER], fontsize=10)
ax.set_title("Regime Centroid Heatmap\n(feature-normalised means)", fontsize=11, fontweight="bold")

for i in range(len(REGIME_ORDER)):
    for j in range(len(TELEMETRY_FEATURES)):
        ax.text(j, i, f"{means_norm[i,j]:.2f}", ha="center", va="center",
                fontsize=8, color="black" if 0.3 < means_norm[i,j] < 0.7 else "white")

plt.colorbar(im, ax=ax, fraction=0.046, label="Normalised value")

fig.suptitle("Behavioral Feature Analysis", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig04_separability_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig04_separability_heatmap.png")


Saved: fig04_separability_heatmap.png


### 4.5 Temporal trajectory of a single agent

Observe how an agent's hidden state and telemetry co-evolve over time.
This reveals the AR(1) smoothing, the regime dwelling times, and the transition sharpness.


In [12]:
AGENT_ID = "agent_0000"
agent_df = df[df["agent_id"] == AGENT_ID].copy().reset_index(drop=True)
T_SHOW   = 500    # show first 500 timesteps for clarity

sub = agent_df.iloc[:T_SHOW]

fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(4, 1, hspace=0.08, figure=fig)

feature_plot_order = ["latency", "entropy", "reward", "error_rate"]
ylabels = ["Latency (ms)", "Entropy (bits)", "Reward (a.u.)", "Error rate"]

for row, (feat, ylabel) in enumerate(zip(feature_plot_order, ylabels)):
    ax = fig.add_subplot(gs[row])

    # Shade background by hidden state
    prev_t, prev_s = 0, sub["hidden_state"].iloc[0]
    for t in range(1, len(sub)):
        s = sub["hidden_state"].iloc[t]
        if s != prev_s or t == len(sub) - 1:
            ax.axvspan(prev_t, t, alpha=0.12, color=PALETTE[prev_s], linewidth=0)
            prev_t, prev_s = t, s

    ax.plot(sub["timestep"], sub[feat], lw=1.2, color="#2c3e50", alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=9)

    if row < len(feature_plot_order) - 1:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Timestep", fontsize=10)

    ax.set_xlim(sub["timestep"].min(), sub["timestep"].max())

legend_patches = [
    mpatches.Patch(color=PALETTE[r], alpha=0.45, label=r.capitalize())
    for r in REGIME_ORDER
]
fig.legend(handles=legend_patches, loc="upper right",
           bbox_to_anchor=(0.99, 0.99), fontsize=9, title="Hidden state")

fig.suptitle(
    f"Temporal Trajectory — {AGENT_ID}  (first {T_SHOW} timesteps)\n"
    "Shading indicates hidden behavioral regime",
    fontsize=12, fontweight="bold",
)
plt.savefig("fig05_temporal_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig05_temporal_trajectory.png")


Saved: fig05_temporal_trajectory.png


### 4.6 PCA — Latent geometric structure

The central hypothesis of LBSM: behavioral regimes occupy *geometrically distinct* regions of a latent manifold.  
PCA provides the first linear approximation of this structure before applying nonlinear manifold methods.


In [13]:
# ============================================================
# Full-Dataset PCA Verification
# ============================================================

print("=" * 60)
print("FULL-DATASET PCA VERIFICATION")
print("=" * 60)

# Full telemetry matrix (40,000 observations)
X_full = df[[f"{f}_z" for f in TELEMETRY_FEATURES]].values

# Fit PCA on full dataset
pca_full = PCA(n_components=3, random_state=SEED)
X_full_pca = pca_full.fit_transform(X_full)

# Explained variance
explained_full = pca_full.explained_variance_ratio_

print("\n=== Full-Dataset PCA Explained Variance ===")
print(
    f"PC1={explained_full[0]:.3f}  "
    f"PC2={explained_full[1]:.3f}  "
    f"PC3={explained_full[2]:.3f}"
)

print(f"Cumulative (3 PCs): {explained_full.sum():.3f}")
print()
print("PC loadings:")
loading_df = pd.DataFrame(
    pca_full.components_.T,
    index=list(TELEMETRY_FEATURES),
    columns=["PC1", "PC2", "PC3"],
)
print(loading_df.round(4).to_string())

FULL-DATASET PCA VERIFICATION

=== Full-Dataset PCA Explained Variance ===
PC1=0.798  PC2=0.053  PC3=0.048
Cumulative (3 PCs): 0.899

PC loadings:
                PC1    PC2    PC3
latency       0.412  0.090  0.326
entropy       0.413 -0.326 -0.242
reward       -0.404  0.520  0.431
memory_usage  0.422 -0.021  0.102
error_rate    0.404  0.021  0.667
action_freq   0.393  0.784 -0.440


In [14]:
# ── Subsample for clarity
df_pca_sample = pd.concat([
    df[df["hidden_state"]==r].sample(n=min(1500, (df["hidden_state"]==r).sum()), random_state=SEED)
    for r in REGIME_ORDER
])

X = df_pca_sample[[f"{f}_z" for f in TELEMETRY_FEATURES]].values
y_labels = df_pca_sample["hidden_state"].values

pca = PCA(n_components=3, random_state=SEED)
X_pca = pca.fit_transform(X)

explained = pca.explained_variance_ratio_
print(f"Variance explained: PC1={explained[0]:.3f}  PC2={explained[1]:.3f}  PC3={explained[2]:.3f}")
print(f"Cumulative (3 PCs): {explained.sum():.3f}")
print()
print("PC loadings:")
loading_df = pd.DataFrame(
    pca.components_.T,
    index=list(TELEMETRY_FEATURES),
    columns=["PC1", "PC2", "PC3"],
)
print(loading_df.round(4).to_string())


Variance explained: PC1=0.779  PC2=0.058  PC3=0.052
Cumulative (3 PCs): 0.889

PC loadings:
                PC1    PC2    PC3
latency       0.418 -0.067  0.361
entropy       0.402 -0.268 -0.322
reward       -0.390  0.306  0.623
memory_usage  0.431 -0.094  0.129
error_rate    0.410 -0.145  0.575
action_freq   0.397  0.895 -0.175


In [15]:
print("\n=== Difference vs Sampled PCA ===")
print(
    f"ΔPC1 = {explained_full[0] - explained[0]:+.3f}\n"
    f"ΔPC2 = {explained_full[1] - explained[1]:+.3f}\n"
    f"ΔPC3 = {explained_full[2] - explained[2]:+.3f}"
)

print("\nInterpretation:")
print(
    "The full-dataset PCA confirms that the dominant latent "
    "geometry remains stable under sampling variation. "
    "Differences relative to the balanced subsample PCA are minor "
    "and arise from class-balanced sampling used for visualization clarity."
)


=== Difference vs Sampled PCA ===
ΔPC1 = +0.019
ΔPC2 = -0.005
ΔPC3 = -0.004

Interpretation:
The full-dataset PCA confirms that the dominant latent geometry remains stable under sampling variation. Differences relative to the balanced subsample PCA are minor and arise from class-balanced sampling used for visualization clarity.


- “PCA visualization was computed on a balanced subsample (up to 1500 observations per regime) for plotting clarity and class balance.”

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
pairs = [(0,1), (0,2), (1,2)]
pair_labels = [("PC1","PC2"), ("PC1","PC3"), ("PC2","PC3")]

for ax, (i, j), (xl, yl) in zip(axes, pairs, pair_labels):
    for regime in REGIME_ORDER:
        mask = y_labels == regime
        ax.scatter(
            X_pca[mask, i], X_pca[mask, j],
            c=PALETTE[regime], alpha=0.25, s=10, linewidths=0,
            label=regime,
        )

    # Mark centroids
    for regime in REGIME_ORDER:
        mask = y_labels == regime
        cx, cy = X_pca[mask, i].mean(), X_pca[mask, j].mean()
        ax.scatter(cx, cy, marker="*", s=260, color=PALETTE[regime],
                   edgecolors="black", linewidths=0.8, zorder=5)

    ax.set_xlabel(f"{xl}  ({explained[i]:.1%})", fontsize=10)
    ax.set_ylabel(f"{yl}  ({explained[j]:.1%})", fontsize=10)
    ax.set_title(f"{xl} vs {yl}", fontsize=11, fontweight="bold")

legend_handles = [
    mpatches.Patch(color=PALETTE[r], label=r.capitalize()) for r in REGIME_ORDER
]
axes[0].legend(handles=legend_handles, fontsize=9)

fig.suptitle(
    "PCA Projections of Telemetry Feature Space\n"
    "(★ = regime centroid  ·  each panel shows a different pair of principal components)",
    fontsize=12, fontweight="bold", y=1.03,
)
plt.tight_layout()
plt.savefig("fig06_pca_projections.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig06_pca_projections.png")


Saved: fig06_pca_projections.png


### 4.7 Pair-plot: six-way feature relationships

Full pair-wise scatter matrix for all six telemetry features,
color-coded by regime. Reveals cross-feature correlations and
multi-dimensional separability.


In [17]:
# Sample to keep pair-plot fast
df_pair = pd.concat([
    df[df["hidden_state"]==r].sample(n=min(600, (df["hidden_state"]==r).sum()), random_state=SEED)
    for r in REGIME_ORDER
])

palette_list = {r: PALETTE[r] for r in REGIME_ORDER}

g = sns.PairGrid(
    df_pair[list(TELEMETRY_FEATURES) + ["hidden_state"]],
    hue="hidden_state",
    hue_order=REGIME_ORDER,
    palette=palette_list,
    diag_sharey=False,
    height=1.9,
)
g.map_upper(sns.scatterplot, alpha=0.20, s=8, linewidth=0)
g.map_lower(sns.kdeplot, fill=True, alpha=0.25, levels=4)
g.map_diag(sns.kdeplot, fill=True, alpha=0.55)
g.add_legend(title="Regime", fontsize=9, title_fontsize=9)

g.figure.suptitle(
    "Pair-Plot: Telemetry Feature Relationships by Behavioral Regime",
    fontsize=12, fontweight="bold", y=1.01,
)
plt.savefig("fig07_pairplot.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: fig07_pairplot.png")


Saved: fig07_pairplot.png


## 5. Structural Verdict

> *"If regions emerge, trajectories form, behavioral geometry appears — then the research direction becomes real."*

The section below programmatically evaluates five evidence criteria for latent structure.


In [18]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, GroupKFold

print("=" * 60)
print("LATENT BEHAVIORAL STRUCTURE — EVIDENCE CHECKLIST")
print("=" * 60)

# ── Criterion 1: PCA variance capture
pc3_var = explained.sum()
criterion1 = pc3_var > 0.60
print(f"\n[{'PASS' if criterion1 else 'FAIL'}] C1. PCA (3 PCs) captures >{60}% variance")
print(f"       Actual: {pc3_var:.1%}")

# ── Criterion 2: Separability ratio of best feature (empirical, see B1)
best_sep = max(sep.values())
criterion2 = best_sep > 5.0
print(f"\n[{'PASS' if criterion2 else 'FAIL'}] C2. Best-feature separability ratio > 5.0")
print(f"       Best feature: {max(sep, key=sep.get)}  ({best_sep:.2f})")

# ── Criterion 3: LDA cross-val accuracy, grouped by agent_id to avoid
# temporal-adjacency leakage across folds (adjacent AR(1) timesteps from the
# same agent would otherwise land in both train and test — see B8).
X_lda      = df[[f"{f}_z" for f in TELEMETRY_FEATURES]].values
y_lda      = df["state_label"].values
groups_lda = df["agent_id"].values

gkf = GroupKFold(n_splits=5)
lda = LinearDiscriminantAnalysis()
cv_scores = cross_val_score(lda, X_lda, y_lda, cv=gkf, groups=groups_lda, scoring="accuracy")
lda_acc = cv_scores.mean()
criterion3 = lda_acc > 0.70
print(f"\n[{'PASS' if criterion3 else 'FAIL'}] C3. LDA 5-fold group (by-agent) CV accuracy > 70%")
print(f"       Actual: {lda_acc:.1%}  (±{cv_scores.std():.1%})")

# ── Criterion 4: Centroid distance ratio (empirical, see B2/B3)
D_c4 = D.copy()
np.fill_diagonal(D_c4, np.nan)
min_centroid_dist = np.nanmin(D_c4)
avg_std_radius = np.linalg.norm(stds_emp.mean(axis=0).values)
separation_ratio = min_centroid_dist / (avg_std_radius + 1e-9)
criterion4 = separation_ratio > 1.5
print(f"\n[{'PASS' if criterion4 else 'FAIL'}] C4. Min centroid distance / avg std-radius > 1.5")
print(f"       Min centroid dist: {min_centroid_dist:.1f}  |  Avg σ-radius: {avg_std_radius:.1f}")
print(f"       Separation ratio:  {separation_ratio:.2f}")

# ── Criterion 5: Anomaly regime discrimination
unstable_error = df.loc[df["hidden_state"]=="unstable", "error_rate"].mean()
stable_error   = df.loc[df["hidden_state"]=="stable",   "error_rate"].mean()
error_ratio    = unstable_error / (stable_error + 1e-9)
criterion5 = error_ratio > 5.0
print(f"\n[{'PASS' if criterion5 else 'FAIL'}] C5. Unstable vs Stable error_rate ratio > 5×")
print(f"       Unstable: {unstable_error:.4f}  |  Stable: {stable_error:.4f}")
print(f"       Ratio: {error_ratio:.1f}×")

# ── Summary
passed = sum([criterion1, criterion2, criterion3, criterion4, criterion5])
print()
print("=" * 60)
print(f"RESULT: {passed}/5 criteria met")
if passed >= 4:
    print(">>> STRONG evidence of latent behavioral structure.")
    print(">>> The paper hypothesis is empirically supported at this stage.")
elif passed >= 2:
    print(">>> MODERATE evidence. Further analysis recommended.")
else:
    print(">>> WEAK evidence. Revisit simulation parameterisation.")
print("=" * 60)


LATENT BEHAVIORAL STRUCTURE — EVIDENCE CHECKLIST

[PASS] C1. PCA (3 PCs) captures >60% variance
       Actual: 88.9%

[FAIL] C2. Best-feature separability ratio > 5.0
       Best feature: memory_usage  (4.51)

[PASS] C3. LDA 5-fold group (by-agent) CV accuracy > 70%
       Actual: 96.2%  (±0.2%)

[FAIL] C4. Min centroid distance / avg std-radius > 1.5
       Min centroid dist: 52.7  |  Avg σ-radius: 61.9
       Separation ratio:  0.85

[PASS] C5. Unstable vs Stable error_rate ratio > 5×
       Unstable: 0.3487  |  Stable: 0.0201
       Ratio: 17.3×

RESULT: 3/5 criteria met
>>> MODERATE evidence. Further analysis recommended.


## 6. Dataset Export

Save the telemetry dataset for downstream notebooks:
- `02_manifold_learning.ipynb` — UMAP/t-SNE on the z-scored feature matrix  
- `03_hmm_inference.ipynb` — Hidden Markov Model regime recovery  
- `04_anomaly_detection.ipynb` — Online anomaly scoring


In [19]:
import os

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Full dataset (with z-scores)
out_path = gen.save(f"{OUTPUT_DIR}/telemetry_n{N_AGENTS}_t{N_TIMESTEPS}.csv", include_z_scores=True)
print(f"Saved telemetry → {out_path}")

# Feature matrix + labels (numpy, for manifold notebooks)
X_full = gen.feature_matrix(z_scored=True)
y_full = gen.labels()
np.save(f"{OUTPUT_DIR}/X_telemetry.npy", X_full)
np.save(f"{OUTPUT_DIR}/y_labels.npy",    y_full)
print(f"Saved X_telemetry.npy  shape={X_full.shape}")
print(f"Saved y_labels.npy     shape={y_full.shape}")


Saved telemetry → /home/sicmundus/Desktop/Dev/Research/lbsm-research/outputs/figures/nb01/data/telemetry_n20_t2000.csv
Saved X_telemetry.npy  shape=(40000, 6)
Saved y_labels.npy     shape=(40000,)


## 7. Summary & Next Steps

### What this notebook established

| Finding                                                  | Evidence                                                                                                                 |
| -------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| **Moderate regime separation emerges in latent space**   | PCA projections reveal partially separable low-dimensional behavioral regions across regimes                             |
| **Behavioral features exhibit discriminative structure** | Fisher separability analysis indicates latency, entropy, and error-related metrics contain meaningful regime information |
| **Linear discriminability is observable**                | Cross-validated LDA accuracy suggests behavioral regimes are moderately separable under linear boundaries                |
| **Temporal behavioral structure is coherent**            | AR(1) emission dynamics and Markov transitions generate smooth temporal trajectories and regime continuity               |
| **Anomalous behavioral regimes are detectable**          | The unstable regime exhibits statistically elevated error behavior, indicating potential anomaly-detection viability     |

 Note: Current results provide moderate evidence for latent behavioral organization in low-dimensional statistical manifolds. Further downstream experiments, manifold analyses, trajectory studies, and robustness evaluations are required to determine the stability, continuity, and generalizability of the observed structures.

### Immediate next steps

1. **`02_manifold_learning.ipynb`** — Apply UMAP and t-SNE to reveal *nonlinear* manifold geometry  
2. **`03_hmm_inference.ipynb`** — Fit a Gaussian-emission HMM and recover the hidden state sequence  
3. **`04_anomaly_detection.ipynb`** — Build an online Mahalanobis-distance anomaly scorer  

### Research significance

> This notebook is the **experimental birth** of the paper.  
> The emergence of geometrically separable behavioral regimes in raw telemetry — without supervision —  
> is the core empirical claim of *"Latent Behavioral State Machines: Manifold Geometry of Adaptive Agent Telemetry."*

---
*Generated by the LBSM research simulation core. All figures archived as `fig0N_*.png`.*
